In [1]:
import pathlib
from pathlib import Path
import sys
import pyarrow.parquet as pq
from DataFile.data_name import DataName
from typing import List

from datetime import datetime

def timestamp_to_datetime(timestamp_ms):
    # 将毫秒时间戳转换为秒
    timestamp_s = timestamp_ms / 1000
    # 转换为datetime对象
    dt = datetime.fromtimestamp(timestamp_s)
    # 格式化为指定字符串
    return dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
import pandas as pd
from pathlib import Path

# 获取 Parquet 文件列表
pfs = list(Path(r"E:\tmp\OKX-BL10-BTC-USDT").glob("*.parquet"))


In [3]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

def calc_spearman_with_rolling_avg(df, price_col='price', indicator_col='indicator', rolling_window=50, price_shift=50):
    """
    计算指标的滚动平均值与未来价格之间的Spearman相关系数。

    :param df: 输入的DataFrame
    :param price_col: 价格列的名称
    :param indicator_col: 指标列的名称
    :param rolling_window: 计算指标移动平均的窗口大小
    :param price_shift: 价格向前移动的期数（预测多远的未来）
    :return: (correlation, p_value)
    """
    # 1. 计算指标的滚动平均值
    indicator_avg_col = f'{indicator_col}_avg_{rolling_window}'
    df[indicator_avg_col] = df[indicator_col].rolling(window=rolling_window).mean()
    
    # 2. 创建位移后的未来价格列
    shifted_price_col = f'shifted_{price_col}_{price_shift}'
    df[shifted_price_col] = df[price_col].shift(-price_shift)
    # df['price_chg'] = (df[shifted_price_col] - df[price_col]) / df[price_col]
    df['price_chg'] = np.log(df[shifted_price_col] / df[price_col])
    
    # 3. 删除包含NaN的行
    # rolling会使前(window-1)行产生NaN，shift会使后(shift)行产生NaN
    valid_data = df[[indicator_avg_col, 'price_chg']].dropna()
    
    # 4. 检查是否有足够的数据进行计算
    if len(valid_data) < 2:
        print("Warning: Not enough valid data points after rolling and shifting to calculate correlation.")
        return None, None
        
    # 5. 计算Spearman相关系数
    correlation, p_value = spearmanr(valid_data[indicator_avg_col], valid_data['price_chg'])
    
    return correlation, p_value

In [4]:
import math

def cal_IR(IC):
    mean = sum(IC) / len(IC)
    squared_diff_sum = sum((x - mean) ** 2 for x in IC)
    std_dev = math.sqrt(squared_diff_sum / len(IC))
    return mean / std_dev

In [5]:
def analysis(indicator, rolling_window_size = 600, price_prediction_shift = 120):
    ICs = []
    pvs = []
    for i in range(len(pfs)):
        t = pq.ParquetFile(pfs[i]).read().to_pandas()
        t['indicator'] = indicator(t)
        correlation, p_value = calc_spearman_with_rolling_avg(
            t, 
            price_col='asks_0_price', 
            indicator_col='indicator', 
            rolling_window=rolling_window_size, 
            price_shift=price_prediction_shift
        )
        ICs.append(correlation)
        pvs.append(p_value)

    print(f"平均IC值: {sum(ICs)/len(ICs):.4f}")
    print(f"IR值: {cal_IR(ICs):.4f}")
    print(f"max p_value: {max(pvs):.4f}")

In [6]:
# analysis(lambda df: (df['bids_0_price'] - df['asks_0_price'])/df['asks_0_price'], 600, 120)

In [7]:
# analysis(lambda t: (t['bids_0_amount'] - t['asks_0_amount']) / (t['bids_0_amount'] + t['asks_0_amount']))

In [8]:
# analysis(lambda t: (t['bids_0_amount'] * t['bids_0_count'] - t['asks_0_amount'] * t['asks_0_count']) / (t['bids_0_amount'] * t['bids_0_count'] + t['asks_0_amount'] * t['asks_0_count']))

In [9]:
def ind(t):
    start = 0
    N = 4
    asks = []
    for i in range(start, N):
        asks.append(t[f'asks_{i}_price'] * t[f'asks_{i}_amount'])
    bids = []
    for i in range(start, N):
        bids.append(t[f'bids_{i}_price'] * t[f'bids_{i}_amount'])
    asks_sum = pd.concat(asks, axis=1).sum(axis=1)
    asks_amount = pd.concat([t[f'asks_{i}_amount'] for i in range(N)], axis=1).sum(axis=1)
    bids_sum = pd.concat(bids, axis=1).sum(axis=1)
    bids_amount = pd.concat([t[f'bids_{i}_amount'] for i in range(N)], axis=1).sum(axis=1)
    return (bids_sum / bids_amount - asks_sum / asks_amount) / (bids_sum / bids_amount + asks_sum / asks_amount)


In [10]:
# analysis(ind)

In [11]:
def get_indicator(indicator, rolling_window=600, price_shift=120) -> pd.DataFrame:
    tmp = []
    price_col = 'asks_0_price'
    for i in range(len(pfs)):
        t = pq.ParquetFile(pfs[i]).read().to_pandas()
        t['ind'] = indicator(t)
        indicator_avg_col = f'ind_avg_{rolling_window}'
        t[indicator_avg_col] = t['ind'].rolling(window=rolling_window).mean()
        
        # 2. 创建位移后的未来价格列
        shifted_price_col = f'shifted_{price_col}_{price_shift}'
        t[shifted_price_col] = t[price_col].shift(-price_shift)
        t['price_chg'] = np.log(t[shifted_price_col] / t[price_col])
        
        # 3. 删除包含NaN的行
        # rolling会使前(window-1)行产生NaN，shift会使后(shift)行产生NaN
        valid_data = t.dropna()

        tmp.append(valid_data[['ts', 'ind', 'price_chg']])
    return pd.concat(tmp, axis=0, ignore_index=True)

In [12]:
t = get_indicator(ind)

In [13]:
# NOTICE: 在进行z-score时，注意是否要引入未来函数！！！
t['z-score'] = (t['ind'] - t['ind'].mean()) / t['ind'].std()

In [14]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

# 假设 t 是你的 DataFrame
# 采样数据（可选）
t_sample = t.sample(frac=0.3, random_state=42)
# t_sample = t
# 预计算直方图
counts, bins = np.histogram(t_sample['z-score'], bins=40)

# 计算 KDE
x_range = np.linspace(t_sample['z-score'].min(), t_sample['z-score'].max(), 100)
kde = gaussian_kde(t_sample['z-score'])
kde_values = kde(x_range)

# 绘制
fig = go.Figure()
fig.add_trace(go.Bar(x=bins[:-1], y=counts, name='Histogram', opacity=0.7))
fig.add_trace(go.Scatter(x=x_range, y=kde_values * counts.sum() * (bins[1] - bins[0]), 
                         name='KDE', opacity=0.5, line=dict(color='red')))
fig.update_layout(
    xaxis_title='Z-Score',
    yaxis_title='频次 / 概率密度',
    showlegend=True,
    bargap=0.1,
    template='plotly_white'
)
fig.show()

In [15]:
t['z-score'].max()

18.77222622425773

In [16]:
import plotly.graph_objects as go
import pandas as pd

# 假设你的dataframe叫df，包含z-score和price_chg列
df = t.sample(frac=0.0001, random_state=123)
# 为了性能优化，使用scattergl而非scatter
fig = go.Figure()

fig.add_trace(
    go.Scattergl(
        x=df['z-score'],
        y=df['price_chg'],
        mode='markers',
        marker=dict(
            size=5,  # 减小点的大小以提高性能
            opacity=0.5,  # 半透明以减少视觉重叠
            color='blue'  # 单一颜色避免复杂计算
        )
    )
)

# 优化布局设置
fig.update_layout(
    title='Z-Score vs Price Change',
    xaxis_title='Z-Score',
    yaxis_title='Price Change',
    # showlegend=False,
    # template='plotly_dark',  # 使用暗主题减少渲染负担，可根据需要调整
    width=800,
    height=600
)

# 优化性能的设置
fig.update_xaxes(automargin=True)
fig.update_yaxes(automargin=True)

# 禁用不必要的交互功能以提高性能
# fig.update_layout(
#     dragmode=False,  # 禁用拖拽
#     hovermode='closest'  # 优化hover性能
# )

# 显示图表
fig.show(
    # config={
    #     'displayModeBar': False,  # 隐藏工具栏
    #     'staticPlot': False  # 保持交互但优化
    # }
)

In [17]:
threshold = 18.7
choose = t[t['z-score'] > threshold]
cost = len(choose) * 0.001
earn = choose['price_chg'].sum()
result = earn - cost
print(f'trades: {len(choose)}')
print(cost, earn, result)

trades: 240
0.24 0.03330849151039166 -0.20669150848960832


In [18]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
t_sample = choose
# 预计算直方图
counts, bins = np.histogram(t_sample['price_chg'], bins=40)

# 计算 KDE
x_range = np.linspace(t_sample['price_chg'].min(), t_sample['price_chg'].max(), 100)
kde = gaussian_kde(t_sample['price_chg'])
kde_values = kde(x_range)

# 绘制
fig = go.Figure()
fig.add_trace(go.Bar(x=bins[:-1], y=counts, name='Histogram', opacity=0.7))
fig.add_trace(go.Scatter(x=x_range, y=kde_values * counts.sum() * (bins[1] - bins[0]), 
                         name='KDE', opacity=0.5, line=dict(color='red')))
fig.update_layout(
    xaxis_title='price_chg',
    yaxis_title='频次 / 概率密度',
    showlegend=True,
    bargap=0.1,
    template='plotly_white'
)
fig.show()

In [19]:
choose = t
cost = len(choose) * 0.001
earn = choose['price_chg'].sum()
result = earn - cost
print(cost, earn, result)

76160.135 28.909364524823093 -76131.22563547517
